# Unit 9 -- two case studies: energy-optimal and time-optimal driving

**4SC000 Optimal Control and Reinforcement Learning**

The pendulum was small and friendly. These two are not, and they break in
*opposite* ways -- which is the point of having both.

Both use the same system, an electric vehicle modelled longitudinally as a
point mass with drag, rolling resistance and road grade:

$$\dot s = v, \qquad m\dot v = F - \tfrac12\rho C_dA\,v|v| - mgC_r - mg\,\gamma(s).$$

The battery is **asymmetric**: delivering mechanical power $P=Fv$ costs
$P/\eta_d$ with $\eta_d=0.90$, recovering it returns only $\eta_r|P|$ with
$\eta_r=0.65$. The round trip is $0.585$, and as we will see that single
number is what makes the first problem interesting at all.

Everything is in `vehicle.py` next to this notebook. Unlike the other three
Unit 9 modules it needs **scipy** as well as numpy: battery energy is not a
sum of squares and can be negative, the trip time is a decision variable, and
both problems carry inequality constraints, so a real SQP solver is required
rather than the hand-rolled Levenberg-Marquardt used for the pendulum.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import vehicle as V

plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

print(f"route   {V.ROUTE_LENGTH:.0f} m in {V.TRIP_TIME:.0f} s "
      f"({3.6*V.CRUISE_SPEED:.0f} km/h average)")
print(f"hill    {V.HILL_HEIGHT:.0f} m spread over {V.HILL_LENGTH:.0f} m, "
      f"starting at {V.HILL_START:.0f} m")
print(f"limits  |F| <= {V.F_MAX:.0f} N,  v <= {V.V_MAX:.0f} m/s")
print(f"battery eta_drive = {V.ETA_DRIVE}, eta_regen = {V.ETA_REGEN}, "
      f"round trip {V.ETA_DRIVE*V.ETA_REGEN:.3f}")

## Case study 1: energy-optimal driving

Cover the route in the fixed trip time, entering and leaving at the average
speed, using as little battery energy as possible.

Before solving anything, it is worth seeing why there is a problem at all.
Suppose regeneration were perfect, $\eta_d=\eta_r=1$. The energy spent is
then the change in kinetic energy, plus the change in potential energy, plus
the dissipation. The route's net elevation change is zero and the end speeds
are fixed, so the first two vanish, and minimizing
$\int(\frac12\rho C_dA v^3 + mgC_r v)\,dt$ at fixed $\int v\,dt$ and fixed
$T$ gives **constant speed**, by convexity.

With a perfect battery, cruise control is already optimal and there is
nothing to compute. Everything interesting comes from $\eta_d\eta_r<1$.

In [ ]:
base = V.eco_constant_speed()
print(f"baseline (constant speed): {base['energy']/1e6:.4f} MJ, "
      f"peak |F| = {base['peak_force']:.0f} N")

eco = V.eco_solve(N=40, scheme="hermite-simpson")
saving = 100 * (base["energy"] - eco["energy"]) / base["energy"]
print(f"energy-optimal (N=40):     {eco['energy']/1e6:.4f} MJ  "
      f"({saving:.1f}% saved)")
print(f"speed range: {3.6*eco['v'].min():.1f} to {3.6*eco['v'].max():.1f} km/h")
print(f"force range: {eco['F'].min():.0f} to {eco['F'].max():.0f} N")

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(8.4, 7.2), sharex=True,
                       gridspec_kw={"height_ratios": [1.0, 1.6, 1.3]})
ss = np.linspace(0, V.ROUTE_LENGTH, 800)
ax[0].fill_between(ss, 0, V.elevation(ss), color="0.75", lw=0)
ax[0].set_ylabel("elevation [m]")
ax[1].plot(base["s"], base["v"] * 3.6, "k--", lw=1.4, label="constant speed")
ax[1].plot(eco["s"], eco["v"] * 3.6, lw=2.0, label="energy-optimal")
ax[1].set_ylabel("speed [km/h]"); ax[1].legend(fontsize=9, loc="lower left")
ax[2].axhline(0, color="0.6", lw=0.8)
ax[2].plot(base["s"], base["F"] / 1e3, "k--", lw=1.4)
ax[2].plot(eco["s"], eco["F"] / 1e3, lw=2.0)
ax[2].set_ylabel("force [kN]"); ax[2].set_xlabel("distance [m]")
for a_ in ax:
    a_.axvspan(V.HILL_START, V.HILL_START + V.HILL_LENGTH, color="C1",
               alpha=0.07, lw=0)
plt.tight_layout(); plt.show()

Read the bottom panel. The baseline has to **brake** on the descent, pushing
energy through a round trip that returns $58.5\%$ of it. The optimal
trajectory never does: it arrives at the crest slower, all but shuts the
motor off, and lets gravity do the accelerating -- converting potential
energy straight into kinetic energy, the one conversion in this vehicle that
is lossless.

Nothing told it to do that. No coasting rule, no "lift off before the crest"
heuristic. Only the model, the constraints and a cost.

In [ ]:
print(f"optimal force stays at or above {eco['F'].min():+.1f} N  "
      f"-> {'never brakes' if eco['F'].min() >= 0 else 'brakes'}")
print(f"baseline force dips to          {base['F'].min():+.1f} N")
for label, s_at in (("start", 0.0), ("foot of climb", V.HILL_START),
                    ("crest", V.HILL_START + V.HILL_LENGTH / 2),
                    ("foot of descent", V.HILL_START + V.HILL_LENGTH)):
    print(f"   {label:16s} v = {3.6*np.interp(s_at, eco['s'], eco['v']):5.1f} km/h,"
          f"  F = {np.interp(s_at, eco['s'], eco['F']):7.1f} N")

### Does the solver's number mean anything?

The same question as in the collocation notebook, and the same answer. The
value an NLP reports is its own discretization's estimate, on its own grid.
`eco_audit` keeps only the force schedule, integrates the true dynamics on a
fine grid, and also asks whether the route was finished at all.

In [ ]:
print(f"  {'scheme':17s} {'N':>3s} {'solver [MJ]':>12s} {'simulated [MJ]':>15s} "
      f"{'misses by':>11s}")
for scheme in ("trapezoidal", "hermite-simpson"):
    for N in (20, 40):
        r = V.eco_solve(N=N, scheme=scheme)
        a = V.eco_audit(r, dt=5e-3)
        print(f"  {scheme:17s} {N:3d} {r['energy']/1e6:12.4f} "
              f"{a['energy']/1e6:15.4f} {a['miss_s']:10.1f} m")

Trapezoidal at $N=20$ reports the lowest energy of the four -- by stopping
some twenty metres short of the finish. A cost that beats the others because
the trajectory is infeasible is the classic way to be fooled by a coarse
grid. Hermite-Simpson at the same $N$ is already close to the converged
answer: fourth-order accuracy earning its keep on a smooth problem.

## Case study 2: time-optimal driving

Same vehicle, now on the flat opening $400$ m, from rest to rest, as fast as
possible. Two things change.

First, the **horizon is a decision variable**. The standard fix is to
normalize time, $\sigma=t/T\in[0,1]$, so that $dx/d\sigma = T f(x,F)$ and
$T$ joins the unknowns.

Second, and more importantly, the solution is **not smooth**. The cost
$T=\int_0^T 1\,dt$ does not depend on $F$, so the Hamiltonian is linear in
the input and Pontryagin's minimum principle puts the optimal force on a
bound: accelerate flat out, then brake flat out, with a single switch.

That collapses the whole problem to **one unknown** -- the switching time --
which a root find pins down to the integrator's accuracy. This is the
*indirect* family from the start of the unit, and it gives the yardstick the
direct methods can be scored against.

In [ ]:
T_ref, t_switch, ref = V.time_optimal_reference(dt=5e-3)
print(f"T*        = {T_ref:.6f} s")
print(f"switch at = {t_switch:.6f} s  ({100*t_switch/T_ref:.1f}% of the way)")
print(f"peak speed= {ref['v'].max():.3f} m/s = {3.6*ref['v'].max():.1f} km/h "
      f"(limit {3.6*V.V_MAX:.0f} km/h, so it never binds)")

In [ ]:
sols = {}
for scheme in ("trapezoidal", "hermite-simpson"):
    for N in (10, 20, 40):
        sols[(scheme, N)] = V.sprint_collocation(N=N, scheme=scheme)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ref["t"], ref["v"] * 3.6, "k-", lw=2.2, label="reference")
for N, c in ((10, "C0"), (40, "C2")):
    r = sols[("trapezoidal", N)]
    ax[0].plot(r["t"], r["v"] * 3.6, c + "o-", ms=3.4, lw=1.2,
               label=f"trapezoidal N={N}")
ax[0].set_xlabel("t [s]"); ax[0].set_ylabel("speed [km/h]"); ax[0].legend(fontsize=8.5)
ax[1].step(ref["t"][:-1], ref["F"] / 1e3, "k-", where="post", lw=2.2)
for N, c in ((10, "C0"), (40, "C2")):
    r = sols[("trapezoidal", N)]
    ax[1].plot(r["t"], r["F"] / 1e3, c + "o-", ms=3.4, lw=1.2)
ax[1].axvline(t_switch, color="C3", ls=":", lw=1.6)
ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("force [kN]")
ax[1].set_title("the switch is what the grid cannot see", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
print(f"  {'scheme':17s} {'N':>3s} {'T [s]':>11s} {'T - T* [s]':>13s}")
for scheme in ("trapezoidal", "hermite-simpson"):
    for N in (10, 20, 40):
        r = sols[(scheme, N)]
        print(f"  {scheme:17s} {N:3d} {r['T']:11.5f} {r['T']-T_ref:+13.5f}")
ms = V.sprint_multiple_shooting(N=40, S=8)
print(f"  {'multiple shooting':17s} {40:3d} {ms['T']:11.5f} {ms['T']-T_ref:+13.5f}")

Every error is **positive**: a grid that cannot represent an instantaneous
switch always loses time.

And the comparison with case study 1 is the whole point of running both. On
the smooth problem, Hermite-Simpson was worth roughly a factor of four in
grid size. Here it is worth a factor of about two in the constant and
**nothing in the rate** -- the measured orders over $N=10\ldots80$ are
$1.94$ for trapezoidal and $1.88$ for Hermite-Simpson. Fourth-order accuracy
is a statement about smooth solutions, and this solution is not one.

The remedy is not a finer grid. It is a better parameterization: make the
switching time itself a variable, which is exactly what the reference above
does with a single unknown.

## What the two case studies leave you with

- **Scaling is not a detail.** In SI units these problems span $10^3$ in the
  decision variables and SLSQP fails outright. Everything in `vehicle.py` is
  solved in non-dimensional variables $s/L$, $v/v_{\max}$, $F/F_{\max}$.
- **Match the quadrature to the scheme.** A Hermite-Simpson defect with a
  trapezoidal cost integral is second-order overall, and the scheme's extra
  accuracy is thrown away.
- **Always simulate the answer.**
- **Smoothness decides which method wins** -- and when the structure of the
  solution is known, use it.